##### IMPORTS & MODULES

In [50]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from data_utils import ISICDataset

##### Initializing Paths, and DataSets

In [51]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {device}")

# Directories
DATA_DIR = "./Data"
IMG_DIR = os.path.join(DATA_DIR, "ISBI2016_ISIC_Part1_Training_Data")
MASK_DIR = os.path.join(DATA_DIR, "ISBI2016_ISIC_Part1_Training_GroundTruth")

# Initialize the Dataset
DATASET = ISICDataset(IMG_DIR, MASK_DIR)

# Split into Train (80%) and Validation (20%)
train_size = int(0.8 * len(DATASET))
val_size = len(DATASET) - train_size

train_dataset, val_dataset = random_split(DATASET, [train_size, val_size])
print(f"Training Data SIZE   : {len(train_dataset)}")
print(f"Validation Data SIZE : {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

Using Device: cuda
Training Data SIZE   : 720
Validation Data SIZE : 180


## U-Net Architecture


In [52]:
# Defining the Architecture

# import torch.nn as nn


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.d1 = DoubleConv(3, 64)
        self.p1 = nn.MaxPool2d(2)

        self.d2 = DoubleConv(64, 128)
        self.p2 = nn.MaxPool2d(2)

        self.d3 = DoubleConv(128, 256)
        self.p3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.u3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.c3 = DoubleConv(512, 256)

        self.u2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.c2 = DoubleConv(256, 128)

        self.u1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.c1 = DoubleConv(128, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))

        b = self.bottleneck(self.p3(d3))

        u3 = self.u3(b)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.c3(u3)

        u2 = self.u2(u3)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.c2(u2)

        u1 = self.u1(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.c1(u1)

        return self.out(u1)  # logits output

##### Segmentation Metrics

In [53]:
# Defining Dice loss, Dice score, and IoU score

# import torch


def dice_loss(pred, target, smooth=1):
    pred = torch.sigmoid(pred)
    pred = pred.view(-1)  # flattens (8, 1, 256, 256) → (N,)
    target = target.view(-1)

    intersection = (pred * target).sum()
    return 1 - (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)


def dice_score(pred, target, smooth=1):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    return (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)


def iou_score(pred, target, smooth=1):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection

    return (intersection + smooth) / (union + smooth)

In [54]:
# Initializing model, optimizer, loss function, and scheduler

# import torch.optim as optim

model = UNet().to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
bce = nn.BCEWithLogitsLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=3, factor=0.5
)

In [49]:
# Training loop with validation, metrics, and model checkpointing

epochs = 50
best_dice = 0

for epoch in range(epochs):

    # ---- TRAIN ----
    model.train()
    train_loss = 0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        loss = bce(preds, masks) + dice_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # ---- VALIDATION ----
    model.eval()
    val_loss = 0
    total_dice = 0
    total_iou = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            preds = model(images)

            loss = bce(preds, masks) + dice_loss(preds, masks)
            val_loss += loss.item()

            total_dice += dice_score(preds, masks).item()
            total_iou += iou_score(preds, masks).item()

    avg_dice = total_dice / len(val_loader)

    # Scheduler step
    scheduler.step(avg_dice)

    # Save best model
    if avg_dice > best_dice:
        best_dice = avg_dice
        torch.save(model.state_dict(), "best_model.pth")

    print(f"""
Epoch {epoch+1}/{epochs}
Train Loss: {train_loss/len(train_loader):.4f}
Val Loss  : {val_loss/len(val_loader):.4f}
Dice Score: {avg_dice:.4f}
IoU Score : {total_iou/len(val_loader):.4f}
""")

KeyboardInterrupt: 